In [1]:
head_pred = "head(X1,X3)"
body_preds = ["body1(X1,X2)", "body2(X2,X3)"]

# Extract variables from body predicates by searching for X? patterns
import re

body_vars = set()
for pred in body_preds:
    body_vars.update(re.findall(r'X\d+', pred))
body_vars = sorted(body_vars)
body_vars

['X1', 'X2', 'X3']

In [6]:
import re
from typing import List

def get_predicate_vars(pred: str) -> List[str]:
    """
    Return sorted unique variables found in a predicate string.
    Example: "body1(X1,X2)" -> ['X1', 'X2']
    """
    vars_found = re.findall(r'X\d+', pred)
    # sort by numeric index to keep a consistent order
    unique_sorted = sorted(set(vars_found), key=lambda v: int(v[1:]) if v[1:].isdigit() else v)
    return unique_sorted

In [7]:
get_predicate_vars("body1(X1,X2)")  # Example usage

['X1', 'X2']

In [2]:
var_value_dict = {}
for var in body_vars:
    var_values = [x for x in range(2)]
    var_value_dict[var] = var_values

var_value_dict

{'X1': [0, 1], 'X2': [0, 1], 'X3': [0, 1]}

In [3]:
# Extract different combinations of variable assignments
import itertools

combinations = list(itertools.product(*var_value_dict.values()))
combinations

[(0, 0, 0),
 (0, 0, 1),
 (0, 1, 0),
 (0, 1, 1),
 (1, 0, 0),
 (1, 0, 1),
 (1, 1, 0),
 (1, 1, 1)]

In [4]:
import problog.program as prolog_program
from problog import get_evaluatable

# load dataset.nl file
with open("dataset.nl", "r") as file:
    bk = file.read()

query = """
query(head(1,0)).
"""

program_input = bk + query

model = prolog_program.PrologString(program_input)
result = get_evaluatable().create_from(model).evaluate()
result

{head(1,0): 1.0}

In [23]:
all_var_pred_samples = []

for comb in combinations:

    variable_values = {var: value for var, value in zip(body_vars, comb)}

    all_preds = body_preds + [head_pred]

    pred_value_dict = {}
    pred_value_dict['variable_assignment'] = variable_values.copy()

    for pred in all_preds:
        pred_vars = get_predicate_vars(pred)
        pred_var_values = [variable_values[var] for var in pred_vars]
        # build up a new query for the predicate with assigned values: pred_name(value1,value2,...)
        pred_name = pred.split('(')[0]
        query_pred = f"{pred_name}({','.join(str(v) for v in pred_var_values)})"
        new_query_pred = f"""
                query({query_pred}).
                """
        program_input = bk + new_query_pred

        model = prolog_program.PrologString(program_input)
        result = get_evaluatable().create_from(model).evaluate()
        # convert result to a dictionary
        result_dict = dict(result)
        # replace the keys with their string representation
        result_dict = {str(key): value for key, value in result_dict.items()}
        # get the truth value for the query predicate which is the last 3 characters of the "result"
        result_truth_value = result_dict.get(query_pred, 0)

        # assignment_str = ', '.join(f"{var}={value}" for var, value in variable_values.items())
        # print(f"Assignment: {assignment_str} => Query: {query_pred} => Result: {result}")

        pred_value_dict[pred] = result_truth_value

    # Append the results for this combination
    all_var_pred_samples.append(pred_value_dict)

all_var_pred_samples

[{'variable_assignment': {'X1': 0, 'X2': 0, 'X3': 0},
  'body1(X1,X2)': 1.0,
  'body2(X2,X3)': 0.0,
  'head(X1,X3)': 1.0},
 {'variable_assignment': {'X1': 0, 'X2': 0, 'X3': 1},
  'body1(X1,X2)': 1.0,
  'body2(X2,X3)': 1.0,
  'head(X1,X3)': 0.0},
 {'variable_assignment': {'X1': 0, 'X2': 1, 'X3': 0},
  'body1(X1,X2)': 0.0,
  'body2(X2,X3)': 0.0,
  'head(X1,X3)': 1.0},
 {'variable_assignment': {'X1': 0, 'X2': 1, 'X3': 1},
  'body1(X1,X2)': 0.0,
  'body2(X2,X3)': 1.0,
  'head(X1,X3)': 0.0},
 {'variable_assignment': {'X1': 1, 'X2': 0, 'X3': 0},
  'body1(X1,X2)': 0.0,
  'body2(X2,X3)': 0.0,
  'head(X1,X3)': 1.0},
 {'variable_assignment': {'X1': 1, 'X2': 0, 'X3': 1},
  'body1(X1,X2)': 0.0,
  'body2(X2,X3)': 1.0,
  'head(X1,X3)': 0.0},
 {'variable_assignment': {'X1': 1, 'X2': 1, 'X3': 0},
  'body1(X1,X2)': 1.0,
  'body2(X2,X3)': 0.0,
  'head(X1,X3)': 1.0},
 {'variable_assignment': {'X1': 1, 'X2': 1, 'X3': 1},
  'body1(X1,X2)': 1.0,
  'body2(X2,X3)': 1.0,
  'head(X1,X3)': 0.0}]

In [24]:
# save to a jsonl file
import json

with open("propositionalization_samples.jsonl", "w") as f:
    for sample in all_var_pred_samples:
        f.write(json.dumps(sample) + "\n")